# UNIVERSIDAD DE LAS FUERZAS ARMADAS "ESPE"
## Elaborado por: Camila Bohórquez, Andrés Cárdenas y Deivis Quispe
## Grupo 3
## Fecha: 04-02-2026


In [29]:
import oracledb
import pandas as pd
try:
    conn = oracledb.connect(
        user="hr",
        password="basededatos",
        dsn="localhost:1521/XE"  
    )
    cursor = conn.cursor()
    cursor.callproc("dbms_output.enable")


except oracledb.Error as e:
    print(f"Error detectado: {e}")

# Laboratorio 9
## Ejercicio 1
Crear un procedimiento llamado ADD_JOB para insertar un nuevo puesto de trabajo en
la tabla JOBS, ingrese el id y el puesto de trabajo mediante parámetros.
a. Ejecute el procedimiento ADD_JOB con el valor para jod_id igual a “IT_DBA” y
el job_title igual a “Database Administrator”


In [2]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE add_job
    (p_id IN jobs.job_id%TYPE, p_title IN jobs.job_title%TYPE)
    IS
    BEGIN
        INSERT INTO jobs
        (job_id, job_title)
        VALUES
        (p_id, p_title);
    END add_job;
    """
    )

    cursor.execute(
    f"""
    BEGIN
        add_job('IT_DBA','Database Administrator');
    END;
    """
    )

    cursor.execute(
    """
    SELECT * FROM jobs
    WHERE job_id = 'IT_DBA'
    """
    )
    resultado = cursor.fetchall()
    for row in resultado:
        print(row)


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

('IT_DBA', 'Database Administrator', None, None)


## Ejercicio 2
Crear un procedimiento llamado DEL_JOB para eliminar un puestro de trabajo de la
tabla JOBS, crear la excepción necesaria cuando un puesto de trabajo no pueda ser
borrado o no exista en la base
a. Llame al procedimiento anterior para eliminar el puesto de trabajo con el
codigo IT_DBA

In [3]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE del_job
    (p_id IN jobs.job_id%TYPE)
    IS
    BEGIN
        DELETE FROM jobs
        WHERE job_id = p_id;
        IF SQL%ROWCOUNT > 0
        THEN DBMS_OUTPUT.PUT_LINE('Eliminado el job: ' || p_id || ' con exito');
        ELSE DBMS_OUTPUT.PUT_LINE('No se pudo eliminar el job: ' || p_id);
        END IF;
    EXCEPTION
        WHEN OTHERS THEN
            DBMS_OUTPUT.PUT_LINE('Ocurrio un error al eliminar');
    END del_job;

    """
    )

    cursor.execute(
    f"""
    BEGIN
        del_job('IT_DBA');
    END;
    """
    )
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

Eliminado el job: IT_DBA con exito


## Ejercicio 3
Crear un procedimiento llamado QUERY_EMP que recupere el salario y job_id de la tabla employees cuando envié como parámetro el código del empleado

In [4]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE query_emp
    (p_emp_id IN employees.employee_id%TYPE, p_salary OUT employees.salary%TYPE, p_job_id OUT employees.job_id%TYPE)
    IS
    BEGIN
        SELECT salary, job_id
        INTO p_salary, p_job_id
        FROM employees
        WHERE employee_id = p_emp_id;
        DBMS_OUTPUT.PUT_LINE('El empleado: ' || TO_CHAR(p_emp_id) || ' tiene un salario de: ' || TO_CHAR(p_salary) || ' y pertenece al job id: ' || p_job_id);
    END query_emp;

    """
    )

    cursor.execute(
    f"""
    DECLARE
        v_salario employees.salary%TYPE;
        v_job_id employees.job_id%TYPE;
    BEGIN
        query_emp(200, v_salario, v_job_id);
    END;
    """
    )
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

El empleado: 200 tiene un salario de: 4400 y pertenece al job id: AD_ASST


## Ejercicio 4
Crear un procedimiento llamado QUERY_DEP que envíe como parámetro el código de un departamento y me devuelva información del número de empleados que existen y cual es el valor total de salarios pagados

In [5]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE query_dep
    (p_dep_id IN departments.department_id%TYPE, p_num_empleados OUT NUMBER, p_salarios_pagados OUT NUMBER)
    IS
    BEGIN
        SELECT COUNT(employee_id), SUM(salary)
        INTO p_num_empleados, p_salarios_pagados
        FROM employees
        WHERE department_id = p_dep_id;
        DBMS_OUTPUT.PUT_LINE('El departamento: ' || TO_CHAR(p_dep_id) || ' tiene total de ' || TO_CHAR(p_num_empleados) || ' empleados y ha pagado en salarios: ' || TO_CHAR(p_salarios_pagados));
    END query_dep;

    """
    )

    cursor.execute(
    f"""
    DECLARE
        v_total_emps NUMBER;
        v_total_salarios NUMBER;
    BEGIN
        query_dep(10, v_total_emps, v_total_salarios);
    END;
    """
    )
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

El departamento: 10 tiene total de 1 empleados y ha pagado en salarios: 4400


## Ejercicio 5
Crear un procedimiento llamado QUERY_CIUDAD que envíe como parámetro el código de un departamento y me devuelva información del nombre de la ciudad y los empleados que trabajan en ese departamento

In [6]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE query_ciudad
    (p_dep_id IN departments.department_id%TYPE, p_ciudad OUT locations.city%TYPE, p_num_emp OUT NUMBER)
    IS
    BEGIN
        SELECT l.city, COUNT(e.employee_id)
        INTO p_ciudad, p_num_emp
        FROM departments d
        LEFT JOIN employees e ON d.department_id = e.department_id
        JOIN locations l ON l.location_id = d.location_id
        WHERE d.department_id = p_dep_id
        GROUP BY l.city;
        DBMS_OUTPUT.PUT_LINE('El departamento: ' || TO_CHAR(p_dep_id) || ' tiene total de ' || TO_CHAR(p_num_emp) || ' empleados y esta en la ciudad: ' || TO_CHAR(p_ciudad));
    END query_ciudad;

    """
    )

    cursor.execute(
    f"""
    DECLARE
        v_ciudad locations.city%TYPE;
        v_empleados NUMBER;
    BEGIN
        query_ciudad(10, v_ciudad, v_empleados);
    END;
    """
    )
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

El departamento: 10 tiene total de 1 empleados y esta en la ciudad: Seattle


# Laboratorio 10
## Ejercicio 1
Crear una función llamada ANNUAL_COMP, que dado como parámetro el código de un empleado me devuelva el total de salario y comisión anual que debería recibir al año el empleado

In [2]:
try:
    p_emp_id = 200
    cursor.execute(
    f"""
    CREATE OR REPLACE FUNCTION annual_comp
    (p_emp_id IN employees.employee_id%TYPE)
    RETURN NUMBER
    IS
    v_salario_anual employees.salary%TYPE;
    v_comision employees.commission_pct%TYPE;
    v_total NUMBER;
    BEGIN
        SELECT salary * 12, NVL(commission_pct,0)
        INTO v_salario_anual, v_comision
        FROM employees
        WHERE employee_id = p_emp_id;
        v_total := v_salario_anual + (v_salario_anual * v_comision);
        RETURN v_total;
    END annual_comp;
    """
    )

    cursor.execute(
    f"""
    BEGIN
        DBMS_OUTPUT.PUT_LINE('El total que gana el empleado ' || TO_CHAR({p_emp_id}) || ' es: ' || TO_CHAR(annual_comp({p_emp_id})));
    END;
    """
    )
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

El total que gana el empleado 200 es: 14808


## Ejercicio 2
Crear un procedimiento llamado NEW_EMP, para ingresar un nuevo empleado en la tabla EMPLOYEES el procedimiento llamara a una función VALID_DEPTID que validará que el código del departamento exista en la tabla DEPARTMENTS

In [8]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE FUNCTION valid_deptid
    (p_dep_codigo IN departments.department_id%TYPE)
        RETURN BOOLEAN
    IS
    v_codigo departments.department_id%TYPE;
    BEGIN
        SELECT department_id
        INTO v_codigo
        FROM departments
        WHERE department_id = p_dep_codigo;
        RETURN TRUE;
    EXCEPTION
        WHEN NO_DATA_FOUND THEN
            RETURN FALSE;
    END valid_deptid;
    """
    )

    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE new_emp
    (p_emp_id IN employees.employee_id%TYPE, 
    p_last_name IN employees.last_name%TYPE, 
    p_email IN employees.email%TYPE,
    p_hire_date IN employees.hire_date%TYPE,
    p_job_id IN employees.job_id%TYPE,
    p_dep_id IN employees.department_id%TYPE
    )
    IS
    BEGIN
        IF valid_deptid(p_dep_id)
        THEN
            INSERT INTO employees
            (employee_id, last_name, email, hire_date, job_id, department_id)
            VALUES 
            (p_emp_id, p_last_name, p_email, p_hire_date, p_job_id, p_dep_id);
            DBMS_OUTPUT.PUT_LINE('Empleado insertado con exito');
        ELSE
            DBMS_OUTPUT.PUT_LINE('No existe el departamento para insertar ese empleado');
        END IF;
    END new_emp;
    """
    )

    cursor.execute(
    """
        BEGIN
            new_emp(2025, 'Cardenas', 'ac@espe.com', SYSDATE, 'IT_PROG', 1);
        END;
    """
    )
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

No existe el departamento para insertar ese empleado


## Ejercicio 3
Crear una función llamada QUERY_JEFE que dado como parámetro el código de un empleado me devuelva el nombre de su jefe inmediato, probar su funcionamiento mediante un select a todos los empleados de la tabla employees

In [9]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE FUNCTION query_jefe
    (p_emp_id IN employees.employee_id%TYPE)
    RETURN VARCHAR2
    IS
        v_jefe_id employees.manager_id%TYPE;
        v_jefe_nombre VARCHAR2(200);
    BEGIN
        SELECT manager_id
        INTO v_jefe_id
        FROM employees
        WHERE employee_id = p_emp_id;
        SELECT first_name || ' ' || last_name
        INTO v_jefe_nombre
        FROM employees
        WHERE employee_id = v_jefe_id;
        RETURN v_jefe_nombre;
    EXCEPTION
        WHEN NO_DATA_FOUND
        THEN RETURN 'No tiene un jefe inmediato';
    END query_jefe;
    """
    )

    cursor.execute(
    f"""
    SELECT first_name || ' ' || last_name AS "Empleado", query_jefe(employee_id) AS "Jefe_inmediato"
    FROM employees
    """
    )
    resultado = cursor.fetchall()
    columnas = ['Empleado', 'Jefe inmediato']
    df = pd.DataFrame(resultado, columns=columnas)
    print(df)


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

            Empleado     Jefe inmediato
0         Ellen Abel      Eleni Zlotkey
1        Sundar Ande  Alberto Errazuriz
2     Mozhe Atkinson         Adam Fripp
3       Shelli Baida             Den Li
4         Amit Banda  Alberto Errazuriz
..               ...                ...
102    Matthew Weiss        Steven King
103  Jennifer Whalen         Neena Yang
104   David Williams    Alexander James
105       Neena Yang        Steven King
106    Eleni Zlotkey        Steven King

[107 rows x 2 columns]


## Ejercicio 4
Crear una función llamada QUERY_DEPT que dado como parámetro el código de un departamento me devuelva el número de puestos de trabajo diferentes que existen en ese departamento

In [10]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE FUNCTION query_dept
    (p_dep_id IN departments.department_id%TYPE)
    RETURN NUMBER
    IS
        v_num_empleados NUMBER;
    BEGIN
        SELECT COUNT(job_id)
        INTO v_num_empleados
        FROM
            (SELECT DISTINCT job_id
            FROM employees
            WHERE department_id = p_dep_id);
        RETURN v_num_empleados;
    END query_dept;
    """
    )

    cursor.execute(
    f"""
    SELECT department_name, query_dept(department_id)
    FROM departments
    """
    )
    resultado = cursor.fetchall()
    columnas = ['Departamento', 'Numero de puestos de trabajo']
    df = pd.DataFrame(resultado, columns=columnas)
    print(df)


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

            Departamento  Numero de puestos de trabajo
0         Administration                             1
1              Marketing                             2
2             Purchasing                             2
3        Human Resources                             1
4               Shipping                             3
5                     IT                             1
6       Public Relations                             1
7                  Sales                             2
8              Executive                             2
9                Finance                             2
10            Accounting                             2
11              Treasury                             0
12         Corporate Tax                             0
13    Control And Credit                             0
14  Shareholder Services                             0
15              Benefits                             0
16         Manufacturing                             0
17        

# Laboratorio 11
## Ejercicio 1:
Crear un trigger a nivel de fila que se dispara "antes" que se ejecute un "update" sobre el
campo "salario" de la tabla " empleados". En el cuerpo del disparador se debe ingresar en
la tabla "control", el nombre del usuario que realizó la actualización, la fecha, el código del
empleado que ha sido modificado, el salaro anterior y el nuevo:

In [11]:
try:    
    cursor.execute(
        """
        CREATE TABLE control (
            usuario         VARCHAR2(50),
            fecha           DATE,
            id_empleado     NUMBER,
            salario_antiguo NUMBER(8,2),
            salario_nuevo   NUMBER(8,2)
        )
        """
    )
except oracledb.Error as e:
    print('Ocurrio un error: ', e)

In [12]:
try:
    cursor.execute(
    """
    CREATE OR REPLACE TRIGGER actualizar_emp
    BEFORE UPDATE ON employees
    FOR EACH ROW
    BEGIN
        IF UPDATING('salary')
        THEN
            INSERT INTO control
            (usuario, fecha, id_empleado, salario_antiguo, salario_nuevo)
            VALUES
            (USER, SYSDATE, :new.employee_id, :old.salary, :new.salary);
        END IF;
    END actualizar_emp;
    """
    )
    cursor.execute(
    """
    UPDATE employees
    SET salary = 1234
    WHERE employee_id = 200
    """
    )

    cursor.execute(
    """
    SELECT * FROM control
    """
    )
    resultado = cursor.fetchall()
    columnas = ['USUARIO', 'FECHA DE CAMBIO', 'ID EMPLEADO', 'SALARIO ANTIGUO', 'SALARIO NUEVO']
    df = pd.DataFrame(resultado, columns = columnas)
    print(df)

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

  USUARIO     FECHA DE CAMBIO  ID EMPLEADO  SALARIO ANTIGUO  SALARIO NUEVO
0     HR2 2026-02-04 05:11:37          200           4400.0         1234.0


## Ejercicio 2
Crear un disparador para múltiples eventos, que se dispare al ejecutar "insert", "update" y
"delete" sobre "empleados". En el cuerpo del trigger se realiza la siguiente acción: se
almacena el nombre del usuario, la fecha y los antiguos y viejos valores de "salario":


In [13]:
try:
    cursor.execute(
    """
    CREATE OR REPLACE TRIGGER disparador_emp
    BEFORE INSERT OR UPDATE OR DELETE ON employees
    FOR EACH ROW
    BEGIN
        IF INSERTING THEN
            INSERT INTO control
            (usuario, fecha, id_empleado, salario_antiguo, salario_nuevo)
            VALUES
            (USER, SYSDATE, :new.employee_id, NULL, :new.salary); 
        ELSIF UPDATING THEN
            INSERT INTO control
            (usuario, fecha, id_empleado, salario_antiguo, salario_nuevo)
            VALUES
            (USER, SYSDATE, :new.employee_id, :old.salary, :new.salary);
        ELSIF DELETING THEN
            INSERT INTO control
            (usuario, fecha, id_empleado, salario_antiguo, salario_nuevo)
            VALUES
            (USER, SYSDATE, :old.employee_id, :old.salary, NULL);
        END IF;
    END;
    """)

    cursor.execute(
    """
    BEGIN
        INSERT INTO employees 
        (employee_id, last_name, email, hire_date, job_id, salary)
        VALUES
        (999, 'Empleado_eliminado', 'el@gmail.com', SYSDATE, 'IT_PROG', 4000);
        
        UPDATE employees 
        SET salary = 4500 
        WHERE employee_id = 999;
        
        DELETE FROM employees 
        WHERE employee_id = 999;
        END;
    """)

    cursor.execute("SELECT * FROM control")
    resultado = cursor.fetchall()
    columnas = ['USUARIO', 'FECHA DE CAMBIO', 'ID EMPLEADO', 'SALARIO ANTIGUO', 'SALARIO NUEVO']
    df = pd.DataFrame(resultado, columns = columnas)
    print(df)

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

  USUARIO     FECHA DE CAMBIO  ID EMPLEADO  SALARIO ANTIGUO  SALARIO NUEVO
0     HR2 2026-02-04 05:11:37          200           4400.0         1234.0
1     HR2 2026-02-04 05:11:50          999              NaN         4000.0
2     HR2 2026-02-04 05:11:50          999           4000.0         4500.0
3     HR2 2026-02-04 05:11:50          999           4000.0         4500.0
4     HR2 2026-02-04 05:11:50          999           4500.0            NaN


crear un procedimiento que llame a una funcion f_localidad y muestre el nombre del empleado y el valor de f_localidad

f_localidad dado un id del empleado da la localidad del empleado

In [53]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE FUNCTION f_localidad
    (p_emp_codigo IN employees.employee_id%TYPE)
    RETURN VARCHAR2
    IS
    v_localidad locations.city%TYPE;
    BEGIN
        SELECT l.city
        INTO v_localidad
        FROM employees e
        JOIN departments d ON e.department_id = d.department_id
        JOIN locations l ON d.location_id = l.location_id
        WHERE e.employee_id = p_emp_codigo;
        RETURN v_localidad;
    END f_localidad;
    """
    )

    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE pro_llamar
    ()
    CURSOR empleados IS
            SELECT employee_id, first_name || ' ' || last_name
            FROM employees;
    IS
        CURSOR empleados IS
            SELECT employee_id, first_name || ' ' || last_name
            FROM employees;
        v_nombre VARCHAR2(100);
        v_localidad VARCHAR2(100);
        v_id NUMBER;
    BEGIN
        OPEN empleados
        LOOP
            FETCH empleados INTO v_id, v_nombre;
            EXIT WHEN empleados%NOTFOUND;
            DBMS_OUTPUT.PUT_LINE(v_nombre || ' ' || f_localidad(v_id));
        END LOOP;
        CLOSE empleados;
    END pro_llamar;
    """
    )
    cursor.execute(
    """
    BEGIN
        pro_llamar;
    END;
    """)
   
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

Ocurrió un error: ORA-06550: line 2, column 9:
PLS-00905: object HR.PRO_LLAMAR is invalid
ORA-06550: line 2, column 9:
PL/SQL: Statement ignored
Help: https://docs.oracle.com/error-help/db/ora-06550/


empleados, nombre del departamento y numero de compañeros de su departamento y el valor del salario pagados a ellos

In [ ]:
import oracledb

try:
    # 1. Función Salario
    cursor.execute("""
    CREATE OR REPLACE FUNCTION salario_dep(p_dep_id IN NUMBER) RETURN NUMBER IS
        v_salario NUMBER;
    BEGIN
        SELECT SUM(salary) INTO v_salario FROM employees WHERE department_id = p_dep_id;
        RETURN NVL(v_salario, 0);
    END;
    """)

    # 2. Función Compañeros
    cursor.execute("""
    CREATE OR REPLACE FUNCTION compa_dep(p_dep_id IN NUMBER) RETURN NUMBER IS
        v_compa NUMBER;
    BEGIN
        SELECT (COUNT(employee_id)-1) INTO v_compa FROM employees WHERE department_id = p_dep_id;
        RETURN CASE WHEN v_compa < 0 THEN 0 ELSE v_compa END;
    END;
    """)

    # 3. Procedimiento con CURSOR EXPLÍCITO
    cursor.execute("""
    CREATE OR REPLACE PROCEDURE proceso IS
        -- 1. DECLARE: Definimos el cursor y las variables para capturar los datos
        CURSOR c_datos IS 
            SELECT e.employee_id, d.department_name, e.department_id
            FROM employees e
            JOIN departments d ON d.department_id = e.department_id;
            
        v_id     employees.employee_id%TYPE;
        v_dept   departments.department_name%TYPE;
        v_did    employees.department_id%TYPE;
    BEGIN
        -- 2. OPEN: Se ejecuta la consulta
        OPEN c_datos;
        
        LOOP
            -- 3. FETCH: Sacamos la fila actual y la metemos en las variables
            FETCH c_datos INTO v_id, v_dept, v_did;
            
            -- Condición de salida: cuando ya no hay más filas
            EXIT WHEN c_datos%NOTFOUND;
            
            -- Imprimimos usando las funciones creadas
            DBMS_OUTPUT.PUT_LINE(
                'Emp: ' || v_id || 
                ' | Dept: ' || v_dept || 
                ' | Compañeros: ' || compa_dep(v_did) || 
                ' | Total Dept: ' || salario_dep(v_did)
            );
        END LOOP;
        
        -- 4. CLOSE: Liberamos la memoria del cursor
        CLOSE c_datos;
    END proceso;
    """)

    # --- EJECUCIÓN ---
    
    # Habilitar salida y ejecutar
    cursor.execute("BEGIN proceso; END;")

    # Recuperar el buffer de DBMS_OUTPUT en Python
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0: break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Error: {e}")

listar los empleados, numero de companieros, y comprobar por cada empleado si el salario esta en los rangos

In [1]:
try:
    cursor.execute(
    """
    CREATE OR REPLACE FUNCTION mensaje
    (p_salario IN NUMBER) 
    RETURN VARCHAR2
    IS
    BEGIN
        RETURN CASE 
            WHEN p_salario < 1500 THEN 'SUBIR'
            WHEN p_salario BETWEEN 1500 AND 2500 THEN 'SALARIO_MEDIO'
            ELSE 'NO SUBIR SALARIO'
        END;
    END mensaje;
    """)

  
    cursor.execute(
    """
    CREATE OR REPLACE FUNCTION compa_dep
    (p_dep_id IN NUMBER) 
    RETURN NUMBER 
    IS
        v_compa NUMBER;
    BEGIN
        SELECT (COUNT(employee_id)-1) INTO v_compa FROM employees WHERE department_id = p_dep_id;
        RETURN CASE WHEN v_compa < 0 THEN 0 ELSE v_compa END;
    END compa_dep;
    """)

   
    cursor.execute(
    """
    CREATE OR REPLACE PROCEDURE proceso IS
        CURSOR empleados IS 
            SELECT e.employee_id, d.department_name, e.department_id, e.salary
            FROM employees e
            JOIN departments d ON d.department_id = e.department_id;
            
        v_emp_id      employees.employee_id%TYPE;
        v_dept_nombre   departments.department_name%TYPE;
        v_dept_id     employees.department_id%TYPE;
        v_salario    employees.salary%TYPE;
    BEGIN
        OPEN empleados;
        LOOP
            FETCH empleados INTO v_emp_id, v_dept_nombre, v_dept_id, v_salario;
            EXIT WHEN empleados%NOTFOUND;
            DBMS_OUTPUT.PUT_LINE(TO_CHAR(v_emp_id) || ' ' || v_dept_nombre || ' ' || TO_CHAR(compa_dep(v_dept_id)) || ' ' || mensaje(v_salario));
        END LOOP;
        CLOSE empleados;
    END proceso;
    """)

    cursor.execute(
    """
    BEGIN
        proceso;
    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0: break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Error: {e}")

NameError: name 'oracledb' is not defined

dado un codigo de empleado cambiar su salario en 100
guardar en una tabla de auditoria el usuario, hora y empleado

In [60]:
try:    
    cursor.execute(
        """
        CREATE TABLE auditoria (
            usuario         VARCHAR2(50),
            fecha           DATE,
            id_empleado     NUMBER
        )
        """
    )
except oracledb.Error as e:
    print('Ocurrio un error: ', e)

In [71]:
try:
    cursor.execute(
    """
    CREATE OR REPLACE TRIGGER actualizar_emp
    BEFORE UPDATE ON employees
    FOR EACH ROW
    BEGIN
        IF UPDATING
        THEN
            INSERT INTO auditoria
            (usuario, fecha, id_empleado)
            VALUES
            (USER, SYSDATE, :old.employee_id);
        END IF;
    END actualizar_emp;
    """
    )
    cursor.execute(
    """
    CREATE OR REPLACE PROCEDURE proceso 
    (p_emp_id IN employees.employee_id%TYPE) 
    IS
    BEGIN
        UPDATE employees
        SET salary = salary + 100
        WHERE employee_id = p_emp_id;
        IF SQL%ROWCOUNT = 0 THEN
            DBMS_OUTPUT.PUT_LINE('Ese usuario no existe');
        END IF;
    EXCEPTION
        WHEN OTHERS THEN
            DBMS_OUTPUT.PUT_LINE('Ese usuario no existe');
    END proceso;
    """)

    cursor.execute(
    """
    BEGIN
        proceso('xd');
    END;
    """
    )

    cursor.execute(
    """
    SELECT * FROM auditoria
    """
    )
    resultado = cursor.fetchall()
    columnas = ['USUARIO', 'FECHA DE CAMBIO', 'ID EMPLEADO']
    df = pd.DataFrame(resultado, columns = columnas)
    print(df)

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

Ocurrió un error: ORA-06502: PL/SQL: numeric or value error: character to number conversion error
ORA-06512: at line 2
Help: https://docs.oracle.com/error-help/db/ora-06502/
